# UCS3564 – NLP | Assignment 4: Neural Machine Translation (English → French)

**Dataset:** Kaggle – *Language Translation (English–French)* (`devicharith/language-translation-englishfrench`)
**Model:** MarianMT (`Helsinki-NLP/opus-mt-en-fr`) – a Transformer **encoder–decoder with (cross-)attention**, SentencePiece/BPE subwords, **fine-tuned on the Kaggle dataset**.
**Evaluation:** BLEU (sacreBLEU) on the held-out test split of the dataset, and on the assignment sentences + 10 extra sentences.

> Runtime → Change runtime type → **GPU (T4)** before running.

In [ ]:
!pip install -q transformers sentencepiece sacremoses sacrebleu kagglehub pandas matplotlib accelerate scikit-learn ipywidgets

In [ ]:
import os, re, glob, random
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import sacrebleu
from IPython.display import display
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
from transformers import MarianMTModel, MarianTokenizer, get_linear_schedule_with_warmup

SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

set_seed()
print("Device:", DEVICE)

In [ ]:
# ----------------------------- Configuration ------------------------------ #
MODEL_NAME   = "Helsinki-NLP/opus-mt-en-fr"
MAX_WORDS    = 30      # keep sentence pairs with at most this many words
MAX_LEN      = 64      # max subword tokens during training
MAX_GEN_LEN  = 128     # max subword tokens when generating
TRAIN_SIZE   = 30000   # number of training pairs used for fine-tuning (raise for better BLEU)
BATCH_SIZE   = 32
EPOCHS       = 1
LEARNING_RATE = 2e-5
EVAL_N       = 500     # test sentences used for BLEU on the dataset

## 1. Load the Kaggle dataset
The cell downloads the dataset with `kagglehub`. If that fails (no internet / Kaggle access), it asks you to upload the CSV manually
(download it from the Kaggle link above).

In [ ]:
csv_files = []
try:
    import kagglehub
    dataset_path = kagglehub.dataset_download("devicharith/language-translation-englishfrench")
    csv_files = glob.glob(os.path.join(dataset_path, "**", "*.csv"), recursive=True)
    print("Downloaded to:", dataset_path)
except Exception as error:
    print("kagglehub download failed:", error)

if not csv_files:
    try:
        from google.colab import files
        print("Please upload the English-French CSV file from Kaggle.")
        uploaded = files.upload()
        csv_files = [name for name in uploaded.keys() if name.endswith(".csv")]
    except Exception as error:
        raise RuntimeError("Place the dataset CSV in the working directory and set csv_files manually.") from error

DATA_CSV = csv_files[0]
print("Using:", DATA_CSV)
raw_df = pd.read_csv(DATA_CSV)
print("Raw shape:", raw_df.shape)
display(raw_df.head())

In [ ]:
# The dataset has one English column and one French column; pick the two text columns by type.
text_columns = [c for c in raw_df.columns if raw_df[c].dtype == object][:2]
assert len(text_columns) == 2, f"Expected two text columns, found {list(raw_df.columns)}"
df = raw_df[text_columns].copy()
df.columns = ["english", "french"]
print("English column:", text_columns[0], "| French column:", text_columns[1])

## 2. Text cleaning
Whitespace is normalised, missing values/duplicates are removed and very long pairs are dropped.
Case, accents and punctuation are **kept** – they carry meaning in translation (e.g. *à/a*, *é*), and the subword tokenizer handles them.

In [ ]:
def clean_text(text):
    """Collapse whitespace; keep case, accents and punctuation."""
    return re.sub(r"\s+", " ", str(text)).strip()

df = df.dropna()
df["english"] = df["english"].apply(clean_text)
df["french"] = df["french"].apply(clean_text)
df = df[(df["english"] != "") & (df["french"] != "")]
df = df.drop_duplicates().reset_index(drop=True)

df["en_words"] = df["english"].str.split().str.len()
df["fr_words"] = df["french"].str.split().str.len()
df = df[(df["en_words"] <= MAX_WORDS) & (df["fr_words"] <= MAX_WORDS)].reset_index(drop=True)
print("Clean shape:", df.shape)

fig, ax = plt.subplots(1, 2, figsize=(10, 3))
df["en_words"].hist(bins=30, ax=ax[0]); ax[0].set_title("English sentence length (words)")
df["fr_words"].hist(bins=30, ax=ax[1]); ax[1].set_title("French sentence length (words)")
plt.tight_layout(); plt.show()

In [ ]:
# 80 / 10 / 10 split with a fixed seed
train_all, temp_df = train_test_split(df, test_size=0.2, random_state=SEED)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=SEED)
train_df = train_all.sample(n=min(TRAIN_SIZE, len(train_all)), random_state=SEED).reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)
print(f"Split (seed={SEED}): full-train={len(train_all)}  used-for-finetuning={len(train_df)}  "
      f"validation={len(val_df)}  test={len(test_df)}")

## 3. Subword (SentencePiece/BPE) tokenization and padding
`MarianTokenizer` splits rare words into subword pieces, so there are no out-of-vocabulary words. Batches are padded and an attention mask is built.

In [ ]:
tokenizer = MarianTokenizer.from_pretrained(MODEL_NAME)

example_en = "The neurosurgeon performed a craniotomy."
pieces = tokenizer.convert_ids_to_tokens(tokenizer(example_en)["input_ids"])
print("Subword pieces:", pieces)

demo = tokenizer(train_df["english"][:3].tolist(), text_target=train_df["french"][:3].tolist(),
                 padding=True, truncation=True, max_length=MAX_LEN, return_tensors="pt")
print("input_ids shape :", tuple(demo["input_ids"].shape))
print("attention_mask  :", demo["attention_mask"].tolist())
print("labels shape    :", tuple(demo["labels"].shape))

In [ ]:
class TranslationDataset(Dataset):
    def __init__(self, frame):
        self.source = frame["english"].tolist()
        self.target = frame["french"].tolist()
    def __len__(self):
        return len(self.source)
    def __getitem__(self, index):
        return self.source[index], self.target[index]

def collate_batch(batch):
    sources, targets = zip(*batch)
    encoded = tokenizer(list(sources), text_target=list(targets), padding=True,
                        truncation=True, max_length=MAX_LEN, return_tensors="pt")
    encoded["labels"][encoded["labels"] == tokenizer.pad_token_id] = -100   # ignore padding in the loss
    return encoded

train_loader = DataLoader(TranslationDataset(train_df), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_batch)
val_loader = DataLoader(TranslationDataset(val_df.sample(n=min(2000, len(val_df)), random_state=SEED)),
                        batch_size=64, shuffle=False, collate_fn=collate_batch)
print("Training batches per epoch:", len(train_loader))

## 4. Model – Transformer encoder–decoder with attention
Marian is an encoder–decoder architecture; the decoder attends to the encoder states through **cross-attention** (the "attention layer" required by the assignment).
We first measure the **pre-trained baseline**, then **fine-tune on the Kaggle dataset** and compare.

In [ ]:
def translate(sentences, model, num_beams=4, batch_size=64):
    """Beam-search translation of a list of English sentences."""
    model.eval()
    outputs = []
    for start in range(0, len(sentences), batch_size):
        batch = [clean_text(s) for s in sentences[start:start + batch_size]]
        encoded = tokenizer(batch, return_tensors="pt", padding=True, truncation=True,
                            max_length=MAX_GEN_LEN).to(DEVICE)
        with torch.no_grad():
            generated = model.generate(**encoded, num_beams=num_beams, max_length=MAX_GEN_LEN)
        outputs.extend(tokenizer.batch_decode(generated, skip_special_tokens=True))
    return outputs

def corpus_bleu_score(hypotheses, references):
    return sacrebleu.corpus_bleu(hypotheses, [references]).score

test_subset = test_df.sample(n=min(EVAL_N, len(test_df)), random_state=SEED).reset_index(drop=True)

baseline_model = MarianMTModel.from_pretrained(MODEL_NAME).to(DEVICE)
baseline_hyps = translate(test_subset["english"].tolist(), baseline_model)
baseline_bleu = corpus_bleu_score(baseline_hyps, test_subset["french"].tolist())
print(f"Baseline (pre-trained, no fine-tuning) BLEU on dataset test split: {baseline_bleu:.2f}")

In [ ]:
# ---------------------------- Fine-tuning loop ----------------------------- #
set_seed()
model = MarianMTModel.from_pretrained(MODEL_NAME).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, int(0.05 * total_steps), total_steps)
use_amp = DEVICE.type == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

def validation_loss(current_model):
    current_model.eval()
    total, batches = 0.0, 0
    with torch.no_grad():
        for batch in val_loader:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=use_amp):
                total += current_model(**batch).loss.item()
            batches += 1
    return total / batches

step_losses = []
for epoch in range(EPOCHS):
    model.train()
    running = 0.0
    for step, batch in enumerate(train_loader, 1):
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=use_amp):
            loss = model(**batch).loss
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer); scaler.update()
        scheduler.step(); optimizer.zero_grad()
        running += loss.item(); step_losses.append(loss.item())
        if step % 100 == 0:
            print(f"epoch {epoch + 1} step {step}/{len(train_loader)}  train loss {running / step:.4f}")
    print(f"Epoch {epoch + 1} done | validation loss = {validation_loss(model):.4f}")

smoothed = pd.Series(step_losses).rolling(50, min_periods=1).mean()
plt.figure(figsize=(6, 3)); plt.plot(smoothed); plt.xlabel("training step"); plt.ylabel("loss")
plt.title("Fine-tuning loss (smoothed)"); plt.show()

model.save_pretrained("nmt_finetuned"); tokenizer.save_pretrained("nmt_finetuned")

## 5. Evaluation on the dataset's held-out test split (BLEU)

In [ ]:
finetuned_hyps = translate(test_subset["english"].tolist(), model)
finetuned_bleu = corpus_bleu_score(finetuned_hyps, test_subset["french"].tolist())

summary = pd.DataFrame({"Model": ["Pre-trained (baseline)", "Fine-tuned on Kaggle data"],
                        "BLEU (test split)": [round(baseline_bleu, 2), round(finetuned_bleu, 2)]})
display(summary)

preview = test_subset.head(8).copy()
preview["model_output"] = finetuned_hyps[:8]
display(preview[["english", "french", "model_output"]])

## 6. The assignment sentences + 10 additional sentences (BLEU against my references)
Includes idioms, **2 long sentences (>15 words)** and **2 rare/domain-specific** sentences, as required.

In [ ]:
EVAL_SET = [
    # (category, English, reference French)
    ("given",   "The weather in Paris is usually cold during January, but it is warm in July.",
                "Le temps à Paris est généralement froid en janvier, mais il fait chaud en juillet."),
    ("given",   "She likes to read books, watch movies, and travel to new places.",
                "Elle aime lire des livres, regarder des films et voyager dans de nouveaux endroits."),
    ("given",   "The new employee works hard every day and completes his tasks on time.",
                "Le nouvel employé travaille dur chaque jour et termine ses tâches à temps."),
    ("simple",  "I would like a cup of coffee, please.", "Je voudrais une tasse de café, s'il vous plaît."),
    ("simple",  "Where is the nearest train station?", "Où est la gare la plus proche ?"),
    ("simple",  "My brother is learning how to play the guitar.", "Mon frère apprend à jouer de la guitare."),
    ("simple",  "The students are preparing for their final exams next week.",
                "Les étudiants se préparent pour leurs examens finaux la semaine prochaine."),
    ("idiom",   "It is raining cats and dogs today.", "Il pleut des cordes aujourd'hui."),
    ("idiom",   "Break a leg at your performance tonight!", "Bonne chance pour ta représentation ce soir !"),
    ("long",    "The doctor said that the patient should rest for at least two weeks before returning to work.",
                "Le médecin a dit que le patient devait se reposer pendant au moins deux semaines avant de retourner au travail."),
    ("long",    "Although the company had faced many financial difficulties last year, it managed to increase its profits and hire fifty new employees this year.",
                "Bien que l'entreprise ait connu de nombreuses difficultés financières l'année dernière, elle a réussi à augmenter ses bénéfices et à embaucher cinquante nouveaux employés cette année."),
    ("rare",    "The neurosurgeon performed a craniotomy to remove the tumor.",
                "Le neurochirurgien a pratiqué une craniotomie pour retirer la tumeur."),
    ("rare",    "The algorithm uses backpropagation to update the weights of the neural network.",
                "L'algorithme utilise la rétropropagation pour mettre à jour les poids du réseau de neurones."),
]

categories = [c for c, _, _ in EVAL_SET]
sources = [s for _, s, _ in EVAL_SET]
references = [r for _, _, r in EVAL_SET]

hyp_base = translate(sources, baseline_model)
hyp_tuned = translate(sources, model)

rows = []
for cat, src, ref, hb, ht in zip(categories, sources, references, hyp_base, hyp_tuned):
    rows.append({"category": cat, "words": len(src.split()), "english": src, "reference": ref,
                 "baseline_output": hb, "baseline_BLEU": round(sacrebleu.sentence_bleu(hb, [ref]).score, 1),
                 "finetuned_output": ht, "finetuned_BLEU": round(sacrebleu.sentence_bleu(ht, [ref]).score, 1)})
results_df = pd.DataFrame(rows)
pd.set_option("display.max_colwidth", 120)
display(results_df[["category", "words", "english", "reference", "finetuned_output", "finetuned_BLEU", "baseline_BLEU"]])

print(f"Corpus BLEU on these {len(sources)} sentences -> baseline: {corpus_bleu_score(hyp_base, references):.2f} | "
      f"fine-tuned: {corpus_bleu_score(hyp_tuned, references):.2f}")
display(results_df.groupby("category")[["baseline_BLEU", "finetuned_BLEU"]].mean().round(1))

### Analysis (check each point against the table above and edit to match your run)
* **Idioms** ("raining cats and dogs", "break a leg") are the usual failure point: a model that translates word-by-word produces *"il pleut des chats et des chiens"*; BLEU is low even when the meaning is partly right because the reference uses a different idiom.
* **Long sentences (>15 words)** test long-range dependencies (agreement, the subjunctive *"ait connu"* after *"bien que"*). Errors typically appear late in the sentence.
* **Rare/domain words** (*craniotomy, backpropagation*) are split into subword pieces by SentencePiece, so they are never out-of-vocabulary, but the translation may be a literal or partial rendering.
* **Fine-tuning effect:** the Kaggle dataset consists mostly of short everyday sentences, so fine-tuning improves BLEU on its own test split while the effect on long/technical sentences can be neutral or slightly negative (compare `baseline_BLEU` vs `finetuned_BLEU`).
* BLEU is a strict n-gram metric: valid paraphrases are penalised, so manual inspection is also needed.

## 7. Bonus tasks
**(b) Attention heatmaps** – decoder cross-attention (last layer, averaged over heads).
**(c) Interactive translator widget.**

In [ ]:
attention_model = MarianMTModel.from_pretrained("nmt_finetuned", attn_implementation="eager").to(DEVICE).eval()

def plot_attention(sentence, title_suffix=""):
    sentence = clean_text(sentence)
    encoded = tokenizer([sentence], return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        generated = attention_model.generate(**encoded, num_beams=1, max_length=64)
        output = attention_model(**encoded, decoder_input_ids=generated, output_attentions=True)
    attention = output.cross_attentions[-1][0].mean(dim=0).float().cpu().numpy()[:-1]   # row i predicts token i+1
    source_tokens = tokenizer.convert_ids_to_tokens(encoded["input_ids"][0])
    target_tokens = tokenizer.convert_ids_to_tokens(generated[0, 1:])
    fig, ax = plt.subplots(figsize=(max(6, len(source_tokens) * 0.55), max(4, len(target_tokens) * 0.4)))
    image = ax.imshow(attention, cmap="viridis", aspect="auto")
    ax.set_xticks(range(len(source_tokens))); ax.set_xticklabels(source_tokens, rotation=90)
    ax.set_yticks(range(len(target_tokens))); ax.set_yticklabels(target_tokens)
    ax.set_xlabel("English (source) tokens"); ax.set_ylabel("French (generated) tokens")
    ax.set_title("Cross-attention " + title_suffix)
    fig.colorbar(image, ax=ax); plt.tight_layout(); plt.show()

plot_attention(sources[0], "- sentence 1")
plot_attention(sources[1], "- sentence 2")

**(d) Comment (edit after viewing your heatmaps):** the heatmap shows which English tokens each French token attends to. For example, adjective/noun reordering
(*"new employee" → "nouvel employé"*) appears as attention crossing the diagonal, and a token that is translated wrongly typically shows diffuse attention
(no single clear source token), which helps explain the error.

In [ ]:
import ipywidgets as widgets
from IPython.display import display

text_box = widgets.Text(placeholder="Type an English sentence", description="English:",
                        layout=widgets.Layout(width="85%"))
button = widgets.Button(description="Translate", button_style="primary")
output_area = widgets.Output()

def on_translate_clicked(_):
    with output_area:
        output_area.clear_output()
        if text_box.value.strip():
            print("French:", translate([text_box.value], model)[0])

button.on_click(on_translate_clicked)
display(text_box, button, output_area)